# 🏦 Banking AI-Agent: Google Colab

This notebook sets up the Banking AI-Agent on Google Colab.
1. **Ollama Backend**
2. **FastAPI API Server**
3. **Streamlit Web Frontend**

### Step 1: Clone or Update the Repository

In [ ]:
import os
import subprocess

try:
    is_git = subprocess.run(['git', 'rev-parse', '--is-inside-work-tree'], capture_output=True, text=True).stdout.strip() == 'true'
except Exception:
    is_git = False

if is_git:
    print("Already in a git repository. Pulling latest changes...")
    !git pull
else:
    if os.path.exists('NLP-Project2'):
        print("NLP-Project2 directory exists. Pulling latest changes...")
        %cd NLP-Project2
        !git pull
    else:
        print("Cloning repository...")
        !git clone https://github.com/TheWallOnFire/NLP-Project2.git
        %cd NLP-Project2

# Ensure we are in Ex3
if os.path.basename(os.getcwd()) == 'NLP-Project2':
    %cd Ex3
elif os.path.basename(os.getcwd()) != 'Ex3':
    if os.path.exists('Ex3'):
        %cd Ex3


### Step 1.5: Train and Load Intent Model
This step will automatically navigate to Ex2, install the necessary dependencies (like Unsloth), train the intent detection model, and then copy the resulting model weights into Ex3 so the agent can use them.

In [ ]:
import os
import subprocess
import shutil

# Get repo root safely
try:
    repo_root = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True).stdout.strip()
except Exception:
    repo_root = os.getcwd()

EX2_DIR = os.path.join(repo_root, "Ex2")
EX3_DIR = os.path.join(repo_root, "Ex3")
MODEL_DIR = os.path.join(EX3_DIR, "models", "intent_model")
os.makedirs(MODEL_DIR, exist_ok=True)

print("--- Running Ex2 Training Pipeline ---")
%cd {EX2_DIR}

# 1. Install Unsloth and dependencies
!pip install --upgrade --no-cache-dir unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes unsloth_zoo
!sudo ldconfig /usr/lib64-nvidia 2>/dev/null || true
!pip install datasets pandas pyyaml

# 2. Train the model
!python scripts/preprocess_data.py --train_size 1000 --test_size 200
!python scripts/train.py --config configs/train.yaml

# 3. Copy to Ex3
src_model = os.path.join(EX2_DIR, "models", "intent_model")
if os.path.exists(src_model):
    shutil.copytree(src_model, MODEL_DIR, dirs_exist_ok=True)
    print("✅ Success: Model trained and copied to Ex3.")
else:
    print("❌ Failed: Model output not found.")

# Return to Ex3 directory
%cd {EX3_DIR}


### Step 2: Install and Start Ollama

In [ ]:
!apt-get update -qq && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import threading
import time

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"], env=env)

threading.Thread(target=run_ollama_serve, daemon=True).start()
time.sleep(5)

### Step 3: Pull the Model
This pulls the **GPT-OSS 20B** model used for response drafting.

In [ ]:
!ollama pull gpt-oss:20b

### Step 4: Install Dependencies

In [ ]:
!pip install -r requirements.txt

### Step 5: Expose Ports via Cloudflare (Stable)
We use **cloudflared** because it is significantly more stable than Localtunnel for Streamlit apps.

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import re

def start_tunnel(port):
    def log_output(process, port):
        for line in iter(process.stdout.readline, b''):
            line = line.decode('utf-8')
            if 'trycloudflare.com' in line:
                url = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
                if url:
                    print(f"\n✅ Port {port} is live at: {url.group(0)}")

    process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    threading.Thread(target=log_output, args=(process, port), daemon=True).start()

print("Starting tunnels...")
start_tunnel(8000) # API
start_tunnel(8501) # Frontend
time.sleep(5)

### Step 6: Launch

In [ ]:
import subprocess
import sys
import threading
import time

# 1. Start the FastAPI Backend with Debug Logging enabled
print("🚀 Starting FastAPI Backend (Debug Logs active)...\n")

def log_backend():
    # Capture output from the backend and print it to the console
    with subprocess.Popen([sys.executable, "run.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as p:
        for line in iter(p.stdout.readline, b''):
            print(f"[BACKEND] {line.decode('utf-8').strip()}")

threading.Thread(target=log_backend, daemon=True).start()
time.sleep(5)

# 2. Start the Streamlit Frontend
print("\n🖥️ Starting Streamlit Frontend...")
!{sys.executable} -m streamlit run frontend/app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false